# 12 — Idempotent Upsert Pattern

Handle inserts + updates safely.

Process schema:

```text
TARGET
+----------+---------+--------+
| event_id | user_id | amount |
+----------+---------+--------+
| e1       | u1      | 10.0   |
| e2       | u2      | 20.0   |
+----------+---------+--------+

INCOMING
+----------+---------+--------+
| event_id | user_id | amount |
+----------+---------+--------+
| e2       | u2      | 25.0   | update
| e3       | u3      | 30.0   | insert
+----------+---------+--------+

UPSERT
  |
  +--> matched keys     = update
  +--> non-matched keys = insert
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("idempotent-upsert")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_patterns_warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## Step 1 — Existing target

In [2]:
schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("amount", DoubleType(), False),
])

target = spark.createDataFrame(
    [
        ("e1", "u1", 10.0),
        ("e2", "u2", 20.0),
    ],
    schema
)

target.show(truncate=False)

+--------+-------+------+
|event_id|user_id|amount|
+--------+-------+------+
|e1      |u1     |10.0  |
|e2      |u2     |20.0  |
+--------+-------+------+



## Step 2 — Incoming batch

In [3]:
incoming = spark.createDataFrame(
    [
        ("e2", "u2", 25.0),
        ("e3", "u3", 30.0),
    ],
    schema
)

incoming.show(truncate=False)

+--------+-------+------+
|event_id|user_id|amount|
+--------+-------+------+
|e2      |u2     |25.0  |
|e3      |u3     |30.0  |
+--------+-------+------+



## Step 3 — Split update vs insert

Avoid ambiguous columns by selecting from the incoming side explicitly.

```text
updates = incoming rows whose key exists in target
inserts = incoming rows whose key does not exist in target
```

In [4]:
target_keys = target.select("event_id")

updates = (
    incoming.alias("i")
    .join(target_keys.alias("t"), on="event_id", how="inner")
    .select("i.event_id", "i.user_id", "i.amount")
)

inserts = (
    incoming.alias("i")
    .join(target_keys.alias("t"), on="event_id", how="left_anti")
    .select("i.event_id", "i.user_id", "i.amount")
)

updates.show(truncate=False)
inserts.show(truncate=False)

+--------+-------+------+
|event_id|user_id|amount|
+--------+-------+------+
|e2      |u2     |25.0  |
+--------+-------+------+

+--------+-------+------+
|event_id|user_id|amount|
+--------+-------+------+
|e3      |u3     |30.0  |
+--------+-------+------+



## Step 4 — Apply upsert

Use aliases and explicit column references.

```text
updated_existing = target rows updated by incoming values when match exists
final = updated_existing + inserts
```

In [5]:
updated_existing = (
    target.alias("t")
    .join(updates.alias("u"), on="event_id", how="left")
    .select(
        F.col("event_id"),
        F.coalesce(F.col("u.user_id"), F.col("t.user_id")).alias("user_id"),
        F.coalesce(F.col("u.amount"), F.col("t.amount")).alias("amount"),
    )
)

final = updated_existing.unionByName(inserts)

final.orderBy("event_id").show(truncate=False)

+--------+-------+------+
|event_id|user_id|amount|
+--------+-------+------+
|e1      |u1     |10.0  |
|e2      |u2     |25.0  |
|e3      |u3     |30.0  |
+--------+-------+------+



## Step 5 — Idempotency check

In [6]:
rerun_inserts = (
    incoming.alias("i")
    .join(final.select("event_id").alias("f"), on="event_id", how="left_anti")
)

print("rows to insert on rerun:", rerun_inserts.count())

rows to insert on rerun: 0


## Step 6 — Control totals

In [7]:
control_totals_schema = StructType([
    StructField("metric", StringType(), False),
    StructField("value", LongType(), False),
])

control_totals = spark.createDataFrame(
    [
        ("target_rows", target.count()),
        ("incoming_rows", incoming.count()),
        ("updates_rows", updates.count()),
        ("inserts_rows", inserts.count()),
        ("final_rows", final.count()),
        ("rerun_insert_rows", rerun_inserts.count()),
    ],
    control_totals_schema
)

control_totals.show(truncate=False)

+-----------------+-----+
|metric           |value|
+-----------------+-----+
|target_rows      |2    |
|incoming_rows    |2    |
|updates_rows     |1    |
|inserts_rows     |1    |
|final_rows       |3    |
|rerun_insert_rows|0    |
+-----------------+-----+



## Final result

```text
TARGET + INCOMING
      |
      v
SPLIT UPDATES / INSERTS
      |
      v
UPDATE EXISTING + UNION INSERTS
      |
      v
RERUN SAFE RESULT
```

In [8]:
spark.stop()